# Webscraping Steam Store for gaming data :

#### <> Installing all the required libraries <>

In [1]:
!pip install requests
!pip install beautifulsoup4
!pip install lxml
!pip install selenium
!pip install webdriver-manager

#### <> Importing the libraries <>

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

### <> Making requests to connect Steam's Web Server to retrieve Source Code <> 

In [3]:
url = "https://store.steampowered.com/search/"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
print(response.status_code)

200


### <> Implementing Selenium for Dynamic Scraping <>

In [4]:
# Launching Chrome.
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
# Opening Steam.
driver.get(url)
# Adding a delay for the page loading.
time.sleep(5)

#### <> Automatic Page Scrolling <>

In [5]:
# Scrolling multiple times.
for i in range(19):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # Scrolling to the bottom.
    print(f"Scrolled {i+1} time(s)")
    time.sleep(3) # Waiting for new games to load.

Scrolled 1 time(s)
Scrolled 2 time(s)
Scrolled 3 time(s)
Scrolled 4 time(s)
Scrolled 5 time(s)
Scrolled 6 time(s)
Scrolled 7 time(s)
Scrolled 8 time(s)
Scrolled 9 time(s)
Scrolled 10 time(s)
Scrolled 11 time(s)
Scrolled 12 time(s)
Scrolled 13 time(s)
Scrolled 14 time(s)
Scrolled 15 time(s)
Scrolled 16 time(s)
Scrolled 17 time(s)
Scrolled 18 time(s)
Scrolled 19 time(s)


#### <> Getting Page Source and Parsing the Web data <>

In [6]:
html = driver.page_source
soup = BeautifulSoup(html, "lxml")
soup

<html class="responsive DesktopUI" lang="en"><head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="width=device-width,initial-scale=1" name="viewport"/>
<meta content="#171a21" name="theme-color"/>
<title>Steam Search</title>
<link href="/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<link href="https://store.fastly.steamstatic.com/public/shared/css/motiva_sans.css?v=YzJgj1FjzW34&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
<link href="https://store.fastly.steamstatic.com/public/shared/css/shared_global.css?v=F0AmfT9E8goz&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
<link href="https://store.fastly.steamstatic.com/public/shared/css/buttons.css?v=BZhNEtESfYSJ&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
<link href="https://store.fastly.steamstatic.com/public/css/v6/store.css?v=QQDgnRsK7TU9&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
<link href="https://store

#### <> Finding game containers <>

In [7]:
games = soup.find_all("a", class_="search_result_row")
print("\nGames found after dynamic scrolling:", len(games))


Games found after dynamic scrolling: 1000


#### <> Extracting the game data columns <>

In [8]:
titles = []
release_dates = []
prices = []
reviews = []
discounts = []
links = []
genre_list = []

for game in games:
# Titles.
    title = game.find("span", class_="title")
    if title:
        title = title.text.strip()
    else:
        title = "N/A"
# Release Dates.
    release = game.find("div", class_="search_released")
    if release:
        release = release.text.strip()
    else:
        release = "N/A"
# Prices.
    price = game.find("div", class_="discount_final_price")
    if price:
        price = price.text.strip()
    else:
        price = "Free"
    titles.append(title)
    release_dates.append(release)
    prices.append(price)
# Reviews.
    review = game.find("span", class_="search_review_summary")
    if review:
        review = review.get("data-tooltip-html")
        if review:
            review = review.replace("<br>", " : ")
        else:
            review = "N/A"
    else:
        review = "N/A"
    reviews.append(review)
# Discounts.
    discount = game.find("div", class_="discount_pct")
    if discount:
        discount = discount.text.strip()
    else:
        discount = "0%"
    discounts.append(discount)
# Links.
    link = game.get("href")
    if link:
        link = link.strip()
    else:
        link = "N/A"
    links.append(link)
# Using Steam Official API to retrieve game genres and Nested data extraction. 
    # 1. Extracting the current app_id.
    app_id = game.get("data-ds-appid")
    if app_id:
        try:
            # 2. Querying the Steam API.
            url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
            response = requests.get(url)
            data = response.json()
            # 3. Extracted nested data safely.
            game_data = data.get(str(app_id))
            if game_data and game_data.get("success"):
                inner_data = game_data.get("data", {})
                genres = inner_data.get("genres", [])
                if genres:
                    genre_names = [genre["description"] for genre in genres if "description" in genre]
                    genre_list.append(", ".join(genre_names))
                else:
                    genre_list.append("Unknown")
            else:
                genre_list.append("Unknown")
        except Exception as e:
            print(f"Error fetching AppID {app_id}: {e}")
            genre_list.append("N/A")
        # This block runs after the try/except finishes, for every different App Id.
        print(f"Done: {app_id}")
        time.sleep(0.2) 
    else:
        genre_list.append("N/A")

Done: 2483190
Done: 730
Done: 1962700
Done: 2215200
Done: 230410
Done: 578080
Done: 1172470
Done: 264710
Done: 1174180
Done: 3321460
Done: 2344520
Done: 2767030
Done: 3892270
Done: 413150
Done: 570
Done: 3472040
Done: 3513350
Done: 3041230
Done: 381210
Done: 3564740
Done: 2399830
Done: 227300
Done: 3124540
Done: 3240220
Done: 848450
Done: 1808500
Done: 252490
Done: 2868840
Done: 3357650
Done: 359550
Done: 236390
Done: 2507950
Done: 3105440
Done: 3241660
Done: 1086940
Done: 1245620
Done: 3405690
Done: 3527290
Done: 2073620
Done: 2807960
Done: 311210
Done: 1295660
Done: 440
Done: 3932890
Done: 553850
Done: 1203220
Done: 3764200
Done: 601150
Done: 270880
Done: 39210
Done: 3404260
Done: 2694490
Done: 3526710
Done: 4197610
Done: 1407200
Done: 440900
Done: 1091500
Done: 3164500
Done: 1973530
Done: 552990
Done: 1222670
Done: 431960
Done: 3672400
Done: 2479810
Done: 1158310
Done: 2357570
Done: 1675200
Done: 1903340
Done: 949230
Done: 281990
Done: 648800
Done: 1151340
Done: 3008130
Done: 114420

#### <> Creating the DataFrame <>

In [9]:
df = pd.DataFrame({ "Title" : titles, "Release_Date" : release_dates, "Price" : prices, "Reviews" : reviews,
"Discount" : discounts, "Game_Link" : links, "Genres" : genre_list })
df.head()

,Title,Release_Date,Price,Reviews,Discount,Game_Link,Genres
0,Forza Horizon 6,"18 May, 2026","₹5,499.00","Very Positive : 86% of the 13,226 user reviews...",0%,https://store.steampowered.com/app/2483190/For...,"Racing, Simulation, Sports"
1,Counter-Strike 2,"21 Aug, 2012",Free,"Very Positive : 86% of the 2,539,502 user revi...",0%,https://store.steampowered.com/app/730/Counter...,"Action, Free To Play"
2,Subnautica 2,"14 May, 2026","₹1,800.00","Very Positive : 93% of the 41,688 user reviews...",0%,https://store.steampowered.com/app/1962700/Sub...,"Action, Adventure, Early Access"
3,LEGO® Batman™: Legacy of the Dark Knight,"22 May, 2026","₹3,899.00",Overwhelmingly Positive : 95% of the 799 user ...,0%,https://store.steampowered.com/app/2215200/LEG...,"Action, Adventure"
4,Warframe,"25 Mar, 2013",Free,"Very Positive : 91% of the 293,180 user review...",0%,https://store.steampowered.com/app/230410/Warf...,"Action, RPG, Free To Play"


In [10]:
df.to_csv("Steam_Games.csv", index=False)

#### <> Closing Chrome <>

In [11]:
driver.quit()